# Australian CPI Forecast 

**Project context:** WaterNSW NSWDF, AUD 3bn, return target CPI + 2.5% p.a. over rolling 10-year period.

**Information cutoff:** 2 April 2026, per project brief. Data truncated to **December quarter 2025**, the last ABS CPI release before the cutoff, published 28 Jan 2026.

## Data sources

| File | Source | Contents |
|---|---|---|
| `6401017.xlsx` | ABS: Consumer Price Index, Australia, Dec 2025 | Table 17: Quarterly All Groups CPI, index numbers, by capital city + Australia |
| `g01hist.xlsx` | RBA Statistical Tables: G1 Consumer Price Inflation | Headline CPI, trimmed mean, weighted median, YoY and QoQ |

Links: [ABS Dec 2025 release](https://www.abs.gov.au/statistics/economy/price-indexes-and-inflation/consumer-price-index-australia/dec-2025#data-downloads) · [RBA Tables](https://www.rba.gov.au/statistics/tables/)

## Series used

| Measure | Series ID | Source | Role |
|---|---|---|---|
| All Groups CPI index level | `A2325846C` | ABS T17 | Long history, 1948 onward, levels modelling |
| Headline CPI YoY % | `GCPIAGYP` | RBA G1 | Forecast target |
| Trimmed mean YoY % | `GCPIOCPMTMYP` | RBA G1 | Underlying signal / state variable |
| Weighted median YoY % | `GCPIOCPMWMYP` | RBA G1 | Robustness check |
| Headline QoQ % SA | `GCPIAGSAQP` | RBA G1 | High-frequency dynamics |
| Trimmed mean QoQ % SA | `GCPIOCPMTMQP` | RBA G1 | Underlying QoQ |

ABS Table 17 headline and RBA G1 headline are the **same series**, RBA sources from ABS. We use G1 for pre-computed YoY and underlying measures, and Table 17 for the long index history.

**Date alignment note:** ABS dates each quarter as the start of the quarter-end month, for example `2025-09-01` for Q3. RBA dates the end of the quarter-end month, for example `2025-09-30`. Both refer to the same quarter. We normalise both to quarter-end before merging.

Trimmed mean and weighted median start around 1983, because RBA did not compute them earlier. This gives around 170 quarterly observations, sufficient for ARIMA / VAR / state-space.

In [27]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Information cutoff per project brief: info available before 2 Apr 2026.
# Last ABS quarterly CPI release before cutoff was Dec quarter 2025.
CUTOFF = '2025-12-31'

ABS Table 17: All Groups CPI Index, Australia

In [28]:
t17 = pd.read_excel('6401017.xlsx', sheet_name='Data1', skiprows=9, index_col=0)
t17.index = pd.to_datetime(t17.index, errors='coerce')
t17 = t17[t17.index.notna()].sort_index().apply(pd.to_numeric, errors='coerce')

# Normalise to quarter-end so ABS and RBA align
t17.index = t17.index.to_period('Q').to_timestamp('Q')

abs_cpi = t17[['A2325846C']].rename(columns={'A2325846C': 'cpi_index'}).loc[:CUTOFF]

print(f"ABS Table 17 — All Groups CPI Index, Australia")
print(f"Date range: {abs_cpi.index.min().date()} to {abs_cpi.index.max().date()}")
print(f"Observations: {len(abs_cpi)}")
print(f"Last value: {abs_cpi['cpi_index'].iloc[-1]:.2f} (should be ~100, Sep 2025 ref period)")
abs_cpi.tail()

ABS Table 17 — All Groups CPI Index, Australia
Date range: 1948-09-30 to 2025-12-31
Observations: 310
Last value: 100.32 (should be ~100, Sep 2025 ref period)


,cpi_index
Series ID,
2024-12-31,96.81
2025-03-31,97.70
2025-06-30,98.43
2025-09-30,99.73
2025-12-31,100.32


RBA G1: headline, trimmed mean, weighted median

Manual loader: series IDs are in row 10, data starts row 11. Building columns explicitly avoids `header=`/`index_col=` ambiguity. `pd.to_numeric(errors='coerce')` prevents silent `object`-dtype failures from stray non-numeric cells.

In [29]:
raw = pd.read_excel('g01hist.xlsx', sheet_name='Data', header=None)

series_ids = raw.iloc[10].tolist()
g1 = raw.iloc[11:].copy()
g1.columns = ['Date'] + series_ids[1:]
g1['Date'] = pd.to_datetime(g1['Date'], errors='coerce')
g1 = g1.dropna(subset=['Date']).set_index('Date').sort_index()
g1.index = g1.index.to_period('Q').to_timestamp('Q')
g1 = g1.apply(pd.to_numeric, errors='coerce').loc[:CUTOFF]

rba = g1[['GCPIAG', 'GCPIAGYP', 'GCPIOCPMTMYP', 'GCPIOCPMWMYP',
          'GCPIAGSAQP', 'GCPIOCPMTMQP']].rename(columns={
    'GCPIAG':       'rba_index',
    'GCPIAGYP':     'headline_yoy',
    'GCPIOCPMTMYP': 'trimmed_mean_yoy',
    'GCPIOCPMWMYP': 'weighted_median_yoy',
    'GCPIAGSAQP':   'headline_qoq_sa',
    'GCPIOCPMTMQP': 'trimmed_mean_qoq_sa',
})

print(f"RBA G1 — Date range: {rba.index.min().date()} to {rba.index.max().date()}")
print(f"Observations: {len(rba)}")
print(f"\nNon-null counts:")
print(rba.notna().sum())
print(f"\nLast 5 obs:")
rba.tail()

RBA G1 — Date range: 1922-06-30 to 2025-12-31
Observations: 415

Non-null counts:
rba_index              415
headline_yoy           411
trimmed_mean_yoy       172
weighted_median_yoy    172
headline_qoq_sa        175
trimmed_mean_qoq_sa    175
dtype: int64

Last 5 obs:


,rba_index,headline_yoy,trimmed_mean_yoy,weighted_median_yoy,headline_qoq_sa,trimmed_mean_qoq_sa
Date,,,,,,
2024-12-31,96.81,2.4,3.3,3.5,0.4,0.5
2025-03-31,97.70,2.4,2.9,3.0,0.9,0.7
2025-06-30,98.43,2.1,2.7,2.8,0.7,0.7
2025-09-30,99.73,3.2,3.0,2.9,1.2,1.0
2025-12-31,100.32,3.6,3.4,3.3,0.8,0.9


In [30]:
df = abs_cpi.join(rba, how='outer').loc[:CUTOFF]
df['abs_yoy_derived'] = df['cpi_index'].pct_change(4, fill_method=None) * 100

# Cross-check on overlapping dates only
overlap = df[['abs_yoy_derived', 'headline_yoy']].dropna()
diff = (overlap['abs_yoy_derived'] - overlap['headline_yoy']).abs()
print(f"Cross-check on {len(overlap)} overlapping observations:")
print(f"  Max |ABS-derived YoY - RBA headline YoY|: {diff.max():.3f}%")
print(f"  (Should be < 0.1% — confirms both sources agree)")
print(f"\nFinal merged dataframe:")
print(f"  Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"  Observations: {len(df)}")
print(f"\nLast 5 obs:")
df[['cpi_index', 'headline_yoy', 'trimmed_mean_yoy',
    'weighted_median_yoy', 'abs_yoy_derived']].tail()

Cross-check on 306 overlapping observations:
  Max |ABS-derived YoY - RBA headline YoY|: 0.050%
  (Should be < 0.1% — confirms both sources agree)

Final merged dataframe:
  Date range: 1922-06-30 to 2025-12-31
  Observations: 415

Last 5 obs:


,cpi_index,headline_yoy,trimmed_mean_yoy,weighted_median_yoy,abs_yoy_derived
2024-12-31,96.81,2.4,3.3,3.5,2.411933
2025-03-31,97.70,2.4,2.9,3.0,2.400168
2025-06-30,98.43,2.1,2.7,2.8,2.095218
2025-09-30,99.73,3.2,3.0,2.9,3.218795
2025-12-31,100.32,3.6,3.4,3.3,3.625659


CPI index level

In [33]:
fig_idx = go.Figure()
fig_idx.add_trace(go.Scatter(x=df.index, y=df['cpi_index'],
                             mode='lines', name='All Groups CPI Index',
                             line=dict(color='#1f77b4', width=2)))
fig_idx.update_layout(title='Australia CPI Index Level (Sep 2025 = 100)',
                      xaxis_title='Quarter', yaxis_title='Index',
                      template='plotly_white', hovermode='x unified')
fig_idx.update_xaxes(range=['1948-01-01', '2026-01-01'])
fig_idx.show()

inflation measures (YoY)

In [32]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['headline_yoy'],
                         mode='lines', name='Headline CPI YoY',
                         line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=df.index, y=df['trimmed_mean_yoy'],
                         mode='lines', name='Trimmed mean YoY',
                         line=dict(color='#d62728', width=2)))
fig.add_trace(go.Scatter(x=df.index, y=df['weighted_median_yoy'],
                         mode='lines', name='Weighted median YoY',
                         line=dict(color='#2ca02c', width=1.5, dash='dot')))

fig.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.2, line_width=0,
              annotation_text='RBA 2-3% target', annotation_position='top right')
fig.add_hline(y=2.5, line_dash='dash', line_color='grey')

fig.update_layout(title='Australian Inflation Measures (data through Dec 2025)',
                  xaxis_title='Quarter', yaxis_title='% change YoY',
                  template='plotly_white', hovermode='x unified',
                  legend=dict(x=0.02, y=0.98))
fig.show()

Working dataframe: `df` (quarterly, indexed on quarter-end dates, through 2025-12-31).

Target: 10-year (40-quarter) forecast of headline CPI YoY, used to define the CPI+2.5% return benchmark for the NSWDF.

Candidate model structures:
- **ARIMA / SARIMA** on headline YoY: univariate baseline
- **VAR** on (headline, trimmed mean, weighted median): captures co-movement
- **State-space** with trimmed mean as latent signal: headline = signal + transitory shock
- **Bayesian VAR** with Minnesota prior shrinking toward 2.5% long-run mean is a defensible long-horizon anchor

# ACF/PACF plot

In [35]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from plotly.subplots import make_subplots

TRAIN_START = '1993-01-01'
y = df.loc[TRAIN_START:CUTOFF, 'headline_yoy'].dropna()
print(f"Training sample: {y.index.min().date()} to {y.index.max().date()}, n={len(y)}\n")

# --- ADF stationarity tests ---
print('Augmented Dickey-Fuller test (H0: unit root, non-stationary)')
for label, series in [('Levels', y), ('First difference', y.diff().dropna())]:
    stat, pval, _, _, crit, _ = adfuller(series, autolag='AIC')
    verdict = 'STATIONARY' if pval < 0.05 else 'non-stationary'
    print(f'  {label:18s}: ADF={stat:7.3f}  p-value={pval:.4f}  -> {verdict}')

# --- ACF / PACF computation ---
NLAGS = 20
ci_band = 1.96 / np.sqrt(len(y))   # 95% confidence band

fig_diag = make_subplots(rows=2, cols=2,
    subplot_titles=('ACF — YoY levels', 'PACF — YoY levels',
                    'ACF — first difference', 'PACF — first difference'))

for col_idx, (label, series) in enumerate([('levels', y), ('diff', y.diff().dropna())]):
    a = acf(series, nlags=NLAGS, fft=False)
    p = pacf(series, nlags=NLAGS, method='ywm')
    lags = np.arange(len(a))
    row = col_idx + 1
    fig_diag.add_trace(go.Bar(x=lags, y=a, marker_color='#1f77b4', showlegend=False), row=row, col=1)
    fig_diag.add_trace(go.Bar(x=lags, y=p, marker_color='#d62728', showlegend=False), row=row, col=2)
    for c in (1, 2):
        fig_diag.add_hline(y=ci_band,  line_dash='dash', line_color='grey', row=row, col=c)
        fig_diag.add_hline(y=-ci_band, line_dash='dash', line_color='grey', row=row, col=c)

fig_diag.update_layout(height=600, template='plotly_white',
                       title='Diagnostics: ACF / PACF (95% CI dashed)')
fig_diag.update_xaxes(title_text='Lag (quarters)')
fig_diag.show()

print('\nReading the plots:')
print('  - ACF tails off slowly + PACF cuts off at lag k  -> AR(k) candidate')
print('  - ACF cuts off at lag k + PACF tails off          -> MA(k) candidate')
print('  - Both tail off                                    -> ARMA(p,q) needed')
print('  - If ADF on levels rejects unit root, d=0 is fine. If only diff is stationary, use d=1.')

Training sample: 1993-03-31 to 2025-12-31, n=132

Augmented Dickey-Fuller test (H0: unit root, non-stationary)
  Levels            : ADF= -3.250  p-value=0.0173  -> STATIONARY
  First difference  : ADF= -6.103  p-value=0.0000  -> STATIONARY



Reading the plots:
  - ACF tails off slowly + PACF cuts off at lag k  -> AR(k) candidate
  - ACF cuts off at lag k + PACF tails off          -> MA(k) candidate
  - Both tail off                                    -> ARMA(p,q) needed
  - If ADF on levels rejects unit root, d=0 is fine. If only diff is stationary, use d=1.


# ARIMA

- Target: `headline_yoy`
- Sample: 1993-Q1 to 2025-Q4 (RBA introduced the 2–3% inflation target in 1993. Before that, mean inflation was structurally higher and more volatile.)
- Validation: train 1993-Q1 to 2015-Q4, forecast 40 quarters, compare to held-out actuals
- Deliverable: refit on full sample, forecast 2026-Q1 to 2035-Q4
- Output: point forecast + 80% / 95% CI, both as YoY and as implied cumulative CPI level

In [36]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from itertools import product
import warnings
warnings.filterwarnings('ignore')

BACKTEST_END = '2015-12-31'
train = df.loc[TRAIN_START:BACKTEST_END, 'headline_yoy'].dropna()
test  = df.loc['2016-01-01':CUTOFF, 'headline_yoy'].dropna()
print(f'Train: {train.index.min().date()} to {train.index.max().date()}, n={len(train)}')
print(f'Test:  {test.index.min().date()}  to {test.index.max().date()}, n={len(test)}')

# --- Grid search over orders, pick lowest AIC ---
p_range, d_range, q_range = range(0, 3), range(0, 2), range(0, 3)
P_range, D_range, Q_range = range(0, 2), range(0, 2), range(0, 2)

results = []
for p, d, q, P, D, Q in product(p_range, d_range, q_range, P_range, D_range, Q_range):
    if p == 0 and q == 0 and P == 0 and Q == 0:
        continue
    try:
        m = SARIMAX(train, order=(p, d, q), seasonal_order=(P, D, Q, 4),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        results.append({'order': (p, d, q), 'seasonal': (P, D, Q, 4), 'aic': m.aic, 'bic': m.bic})
    except Exception:
        continue

results_df = pd.DataFrame(results).sort_values('aic').reset_index(drop=True)
print('\nTop 5 models by AIC:')
print(results_df.head().to_string(index=False))

best_order    = results_df.iloc[0]['order']
best_seasonal = results_df.iloc[0]['seasonal']
print(f'\nBest model: SARIMAX{best_order}x{best_seasonal}')

# --- Refit best, forecast 40 quarters ---
best_model = SARIMAX(train, order=best_order, seasonal_order=best_seasonal,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

fc_obj = best_model.get_forecast(steps=len(test))
fc_mean = fc_obj.predicted_mean
fc_ci80 = fc_obj.conf_int(alpha=0.20)
fc_ci95 = fc_obj.conf_int(alpha=0.05)
fc_idx  = test.index

# --- Metrics ---
actual = test.values
predicted = fc_mean.values
rmse = np.sqrt(np.mean((actual - predicted)**2))
mae  = np.mean(np.abs(actual - predicted))
mape = np.mean(np.abs((actual - predicted) / actual)) * 100
print(f'\nBacktest metrics over {len(test)} quarters:')
print(f'  RMSE: {rmse:.3f} pp')
print(f'  MAE:  {mae:.3f} pp')
print(f'  MAPE: {mape:.1f}%')

# --- Plot ---
fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(x=train.index, y=train.values,
                            mode='lines', name='Train (1993-2015)', line=dict(color='#1f77b4')))
fig_bt.add_trace(go.Scatter(x=test.index, y=test.values,
                            mode='lines', name='Actual (held-out)', line=dict(color='black', width=2)))
fig_bt.add_trace(go.Scatter(x=fc_idx, y=predicted,
                            mode='lines', name='Forecast', line=dict(color='#d62728', dash='dash')))
fig_bt.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci95.iloc[:, 1]) + list(fc_ci95.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.15)', line=dict(width=0),
                            name='95% CI', showlegend=True))
fig_bt.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci80.iloc[:, 1]) + list(fc_ci80.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.25)', line=dict(width=0),
                            name='80% CI', showlegend=True))
fig_bt.add_hline(y=2.5, line_dash='dot', line_color='grey',
                 annotation_text='RBA target midpoint')
fig_bt.update_layout(
    title=f'Backtest: SARIMAX{best_order}x{best_seasonal} — train 1993-2015, forecast 2016-2025',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_bt.show()

Train: 1993-03-31 to 2015-12-31, n=92
Test:  2016-03-31  to 2025-12-31, n=40

Top 5 models by AIC:
    order     seasonal        aic        bic
(0, 1, 2) (0, 0, 1, 4) 154.428522 164.151789
(0, 1, 1) (0, 0, 1, 4) 154.825928 162.153882
(1, 1, 0) (0, 0, 1, 4) 155.051485 162.414527
(1, 1, 1) (0, 0, 1, 4) 155.702871 165.473476
(0, 1, 2) (1, 0, 1, 4) 156.423707 168.577791

Best model: SARIMAX(0, 1, 2)x(0, 0, 1, 4)

Backtest metrics over 40 quarters:
  RMSE: 1.909 pp
  MAE:  1.446 pp
  MAPE: 80.7%


In [40]:
# --- Refit best model on full sample 1993-Q1 to 2025-Q4 ---
y_full = df.loc[TRAIN_START:CUTOFF, 'headline_yoy'].dropna()
print(f'Full training sample: {y_full.index.min().date()} to {y_full.index.max().date()}, n={len(y_full)}')

final_model = SARIMAX(y_full, order=best_order, seasonal_order=best_seasonal,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f'Model: SARIMAX{best_order}x{best_seasonal}, AIC={final_model.aic:.2f}')

# --- In-sample fitted values ---
arima_fitted = final_model.fittedvalues
arima_resid  = y_full - arima_fitted

warmup = max(best_order[1], best_seasonal[1] * 4) + 1
arima_fitted = arima_fitted.iloc[warmup:]
arima_resid  = arima_resid.iloc[warmup:]

rmse_in = np.sqrt(np.mean(arima_resid**2))
mae_in  = np.mean(np.abs(arima_resid))
print(f'In-sample fit (excl. {warmup} warmup obs): RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')

# --- Forecast 40 quarters ahead ---
HORIZON = 40
fc_obj  = final_model.get_forecast(steps=HORIZON)
fc_mean = fc_obj.predicted_mean
fc_ci80 = fc_obj.conf_int(alpha=0.20)
fc_ci95 = fc_obj.conf_int(alpha=0.05)

fc_idx = pd.date_range(start=y_full.index[-1] + pd.offsets.QuarterEnd(),
                       periods=HORIZON, freq='QE-DEC')
fc_mean.index = fc_idx
fc_ci80.index = fc_idx
fc_ci95.index = fc_idx

# --- Summary stats ---
print(f'\nForecast horizon: {fc_idx[0].date()} to {fc_idx[-1].date()}')
print(f'Mean YoY forecast over 10 years: {fc_mean.mean():.2f}%')
print(f'Forecast at end of horizon:      {fc_mean.iloc[-1]:.2f}%')
print(f'Implied CPI+2.5% target return:  {fc_mean.mean() + 2.5:.2f}% p.a.')

# --- Plot: actual + in-sample fit + forecast + CI ---
fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=y_full.index, y=y_full.values,
                            mode='lines', name='Actual',
                            line=dict(color='#1f77b4', width=2)))
fig_fc.add_trace(go.Scatter(x=arima_fitted.index, y=arima_fitted.values,
                            mode='lines', name='In-sample fit',
                            line=dict(color='#d62728', width=1.5, dash='dot')))
fig_fc.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values,
                            mode='lines', name='Forecast (mean)',
                            line=dict(color='#d62728', width=2, dash='dash')))
fig_fc.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci95.iloc[:, 1]) + list(fc_ci95.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.12)',
                            line=dict(width=0), name='95% CI'))
fig_fc.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci80.iloc[:, 1]) + list(fc_ci80.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.22)',
                            line=dict(width=0), name='80% CI'))
fig_fc.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                 annotation_text='RBA 2-3% target', annotation_position='top right')
fig_fc.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_fc.update_layout(
    title=f'10-year CPI YoY Forecast — SARIMAX{best_order}x{best_seasonal} (in-sample fit + forecast)',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fc.show()

# --- Implied CPI level path ---
hist_idx = df['cpi_index'].dropna()
level_path = list(hist_idx.iloc[-4:].values)
for yoy in fc_mean.values:
    level_path.append(level_path[-4] * (1 + yoy / 100))
level_fc = pd.Series(level_path[4:], index=fc_idx)

fig_lvl = go.Figure()
fig_lvl.add_trace(go.Scatter(x=hist_idx.index, y=hist_idx.values,
                             mode='lines', name='Historical CPI',
                             line=dict(color='#1f77b4', width=2)))
fig_lvl.add_trace(go.Scatter(x=level_fc.index, y=level_fc.values,
                             mode='lines', name='Forecast CPI (mean)',
                             line=dict(color='#d62728', dash='dash', width=2)))
fig_lvl.update_layout(
    title='Implied CPI Index Path — 10-year forecast (Sep 2025 = 100)',
    xaxis_title='Quarter', yaxis_title='Index level',
    template='plotly_white', hovermode='x unified')
fig_lvl.show()

# --- Save forecast for downstream use ---
forecast_out = pd.DataFrame({
    'yoy_mean':  fc_mean.values,
    'yoy_lo80':  fc_ci80.iloc[:, 0].values,
    'yoy_hi80':  fc_ci80.iloc[:, 1].values,
    'yoy_lo95':  fc_ci95.iloc[:, 0].values,
    'yoy_hi95':  fc_ci95.iloc[:, 1].values,
    'cpi_level': level_fc.values,
}, index=fc_idx)
print('\nForecast dataframe saved as `forecast_out`. First and last rows:')
print(forecast_out.head(2).round(2))
print(forecast_out.tail(2).round(2))

Full training sample: 1993-03-31 to 2025-12-31, n=132
Model: SARIMAX(0, 1, 2)x(0, 0, 1, 4), AIC=236.71
In-sample fit (excl. 2 warmup obs): RMSE=0.603 pp, MAE=0.420 pp

Forecast horizon: 2026-03-31 to 2035-12-31
Mean YoY forecast over 10 years: 2.78%
Forecast at end of horizon:      2.72%
Implied CPI+2.5% target return:  5.28% p.a.



Forecast dataframe saved as `forecast_out`. First and last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  cpi_level
2026-03-31      3.72      2.97      4.47      2.57      4.87     101.34
2026-06-30      3.47      2.28      4.66      1.65      5.29     101.84
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  cpi_level
2035-09-30      2.72      0.82      4.63     -0.19      5.64     130.80
2035-12-31      2.72      0.82      4.63     -0.19      5.64     131.34


| Column | Meaning |
|---|---|
| `yoy_mean` | Point forecast of headline CPI YoY for that quarter. For example, 2026-Q1 = 3.72%. |
| `yoy_lo80` / `yoy_hi80` | 80% confidence interval, i.e. the model thinks there is an 80% probability that actual YoY CPI for that quarter falls in this range. |
| `yoy_lo95` / `yoy_hi95` | Same as above, but for the 95% interval. This interval is wider. |
| `cpi_level` | Implied CPI index level if the mean forecast plays out exactly. Starts at 101.34 in 2026-Q1 and climbs to 131.34 by 2035-Q4, implying cumulative inflation of about 31% over 10 years. |

# SARIMAX-X with trimmed mean exogenous (robustness check)

Same target (`headline_yoy`), same training window (1993-Q1 to 2025-Q4), but adds **trimmed mean YoY** as an exogenous regressor. Economic logic: headline = underlying signal (trimmed mean) + transitory shocks (fuel, fresh food, regulated prices). Including trimmed mean explicitly should improve the near-term forecast.

For multi-step forecasting we need future trimmed-mean values. We forecast it with its own univariate ARIMA first, then feed those forecasts in as exogenous to the headline model.

**Sample note:** trimmed mean starts 1983, so the merged sample is unchanged from the ARIMA cells — 1993-Q1 onwards (n=132).

In [38]:
# ===== SARIMAX-X backtest: 1993-2015 train, 2016-2025 test =====

# Step 1 — forecast trimmed mean separately (small ARIMA, AIC-selected)
tm_train = df.loc[TRAIN_START:BACKTEST_END, 'trimmed_mean_yoy'].dropna()
tm_test  = df.loc['2016-01-01':CUTOFF, 'trimmed_mean_yoy'].dropna()

tm_results = []
for p, d, q in product(range(0, 3), range(0, 2), range(0, 3)):
    if p == 0 and q == 0: continue
    try:
        m = SARIMAX(tm_train, order=(p, d, q), seasonal_order=(0, 0, 1, 4),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        tm_results.append({'order': (p, d, q), 'aic': m.aic})
    except Exception:
        continue
tm_best_order = pd.DataFrame(tm_results).sort_values('aic').iloc[0]['order']
print(f'Trimmed mean ARIMA: {tm_best_order}x(0,0,1,4)')

tm_model = SARIMAX(tm_train, order=tm_best_order, seasonal_order=(0, 0, 1, 4),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
tm_forecast = tm_model.get_forecast(steps=len(tm_test)).predicted_mean.values

# Step 2 — fit headline SARIMAX-X using actual trimmed mean as exog over training
hl_train = df.loc[TRAIN_START:BACKTEST_END, 'headline_yoy'].dropna()
hl_test  = df.loc['2016-01-01':CUTOFF, 'headline_yoy'].dropna()
exog_train = df.loc[hl_train.index, 'trimmed_mean_yoy'].values.reshape(-1, 1)

# Grid search over headline orders, trimmed mean is in exog
hlx_results = []
for p, d, q, P, D, Q in product(range(0, 3), range(0, 2), range(0, 3),
                                 range(0, 2), range(0, 2), range(0, 2)):
    if p == 0 and q == 0 and P == 0 and Q == 0: continue
    try:
        m = SARIMAX(hl_train, exog=exog_train, order=(p, d, q),
                    seasonal_order=(P, D, Q, 4),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        hlx_results.append({'order': (p, d, q), 'seasonal': (P, D, Q, 4), 'aic': m.aic})
    except Exception:
        continue
hlx_df = pd.DataFrame(hlx_results).sort_values('aic').reset_index(drop=True)
print('\nTop 5 SARIMAX-X models by AIC:')
print(hlx_df.head().to_string(index=False))

best_x_order    = hlx_df.iloc[0]['order']
best_x_seasonal = hlx_df.iloc[0]['seasonal']

# Step 3 — refit best, forecast headline using forecasted trimmed mean as exog
best_x = SARIMAX(hl_train, exog=exog_train, order=best_x_order,
                 seasonal_order=best_x_seasonal,
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

exog_test = tm_forecast.reshape(-1, 1)
fcx_obj = best_x.get_forecast(steps=len(hl_test), exog=exog_test)
fcx_mean = fcx_obj.predicted_mean
fcx_ci80 = fcx_obj.conf_int(alpha=0.20)
fcx_ci95 = fcx_obj.conf_int(alpha=0.05)

# Step 4 — metrics, compare to baseline ARIMA
actual = hl_test.values
predicted = fcx_mean.values
rmse_x = np.sqrt(np.mean((actual - predicted)**2))
mae_x  = np.mean(np.abs(actual - predicted))
print(f'\nSARIMAX-X backtest: RMSE={rmse_x:.3f} pp, MAE={mae_x:.3f} pp')
print(f'Baseline ARIMA:     RMSE={rmse:.3f} pp, MAE={mae:.3f} pp')
print(f'Improvement:        RMSE Δ={rmse - rmse_x:+.3f} pp, MAE Δ={mae - mae_x:+.3f} pp')

# Step 5 — overlay both backtests on one chart
fig_btx = go.Figure()
fig_btx.add_trace(go.Scatter(x=hl_train.index, y=hl_train.values, mode='lines',
                             name='Train', line=dict(color='#1f77b4')))
fig_btx.add_trace(go.Scatter(x=hl_test.index, y=hl_test.values, mode='lines',
                             name='Actual', line=dict(color='black', width=2)))
fig_btx.add_trace(go.Scatter(x=hl_test.index, y=predicted, mode='lines',
                             name='SARIMAX-X forecast',
                             line=dict(color='#d62728', dash='dash', width=2)))
fig_btx.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values, mode='lines',
                             name='Baseline ARIMA forecast',
                             line=dict(color='#ff7f0e', dash='dot', width=2)))
fig_btx.add_trace(go.Scatter(x=list(hl_test.index) + list(hl_test.index[::-1]),
                             y=list(fcx_ci95.iloc[:, 1]) + list(fcx_ci95.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.12)',
                             line=dict(width=0), name='95% CI (SARIMAX-X)'))
fig_btx.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_btx.update_layout(
    title=f'Backtest comparison: SARIMAX-X{best_x_order}x{best_x_seasonal} vs baseline',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_btx.show()

Trimmed mean ARIMA: (2, 1, 0)x(0,0,1,4)

Top 5 SARIMAX-X models by AIC:
    order     seasonal        aic
(2, 0, 1) (0, 0, 1, 4) 140.150685
(1, 0, 2) (0, 0, 1, 4) 140.734659
(1, 0, 1) (0, 0, 1, 4) 141.482667
(2, 0, 0) (0, 0, 1, 4) 141.544142
(2, 0, 2) (0, 0, 1, 4) 141.580664

SARIMAX-X backtest: RMSE=1.950 pp, MAE=1.487 pp
Baseline ARIMA:     RMSE=1.909 pp, MAE=1.446 pp
Improvement:        RMSE Δ=-0.040 pp, MAE Δ=-0.041 pp


In [41]:
# ===== SARIMAX-X deliverable: refit on full sample, 10-year forecast =====

# Step 1 — forecast trimmed mean over the next 40 quarters
tm_full = df.loc[TRAIN_START:CUTOFF, 'trimmed_mean_yoy'].dropna()
tm_final = SARIMAX(tm_full, order=tm_best_order, seasonal_order=(0, 0, 1, 4),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
tm_future = tm_final.get_forecast(steps=HORIZON).predicted_mean.values
print(f'Trimmed mean 10-yr forecast: mean {tm_future.mean():.2f}%, '
      f'end-of-horizon {tm_future[-1]:.2f}%')

# Step 2 — refit headline SARIMAX-X on full sample
exog_full = df.loc[y_full.index, 'trimmed_mean_yoy'].values.reshape(-1, 1)
final_x = SARIMAX(y_full, exog=exog_full, order=best_x_order,
                  seasonal_order=best_x_seasonal,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f'SARIMAX-X{best_x_order}x{best_x_seasonal}, AIC={final_x.aic:.2f}  '
      f'(baseline ARIMA AIC={final_model.aic:.2f})')

# Step 3 — in-sample fitted values
sarimax_fitted = final_x.fittedvalues
sarimax_resid  = y_full - sarimax_fitted

warmup_x = max(best_x_order[1], best_x_seasonal[1] * 4) + 1
sarimax_fitted = sarimax_fitted.iloc[warmup_x:]
sarimax_resid  = sarimax_resid.iloc[warmup_x:]

rmse_in_x = np.sqrt(np.mean(sarimax_resid**2))
mae_in_x  = np.mean(np.abs(sarimax_resid))
print(f'In-sample fit (excl. {warmup_x} warmup obs): RMSE={rmse_in_x:.3f} pp, MAE={mae_in_x:.3f} pp')
print(f'  Baseline ARIMA in-sample: RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')

# Step 4 — forecast 40 quarters with forecasted trimmed mean as exog
exog_future = tm_future.reshape(-1, 1)
fcx_obj  = final_x.get_forecast(steps=HORIZON, exog=exog_future)
fcx_mean = fcx_obj.predicted_mean
fcx_ci80 = fcx_obj.conf_int(alpha=0.20)
fcx_ci95 = fcx_obj.conf_int(alpha=0.05)

fcx_idx = pd.date_range(start=y_full.index[-1] + pd.offsets.QuarterEnd(),
                        periods=HORIZON, freq='QE-DEC')
fcx_mean.index = fcx_idx
fcx_ci80.index = fcx_idx
fcx_ci95.index = fcx_idx

# Step 5 — summary side-by-side
print(f'\n--- 10-year forecast comparison ---')
print(f'Baseline ARIMA mean:   {fc_mean.mean():.2f}%   end:{fc_mean.iloc[-1]:.2f}%')
print(f'SARIMAX-X mean:        {fcx_mean.mean():.2f}%   end:{fcx_mean.iloc[-1]:.2f}%')
print(f'Implied target return (ARIMA):    CPI+2.5% = {fc_mean.mean() + 2.5:.2f}% p.a.')
print(f'Implied target return (SARIMAX-X): CPI+2.5% = {fcx_mean.mean() + 2.5:.2f}% p.a.')

# Step 6 — plot: actual + in-sample fit + both forecasts + CI
fig_fcx = go.Figure()
fig_fcx.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                             name='Actual', line=dict(color='#1f77b4', width=2)))
fig_fcx.add_trace(go.Scatter(x=sarimax_fitted.index, y=sarimax_fitted.values, mode='lines',
                             name='In-sample fit',
                             line=dict(color='#d62728', width=1.5, dash='dot')))
fig_fcx.add_trace(go.Scatter(x=fcx_idx, y=fcx_mean.values, mode='lines',
                             name='SARIMAX-X (forecast)',
                             line=dict(color='#d62728', dash='dash', width=2)))
fig_fcx.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values, mode='lines',
                             name='Baseline ARIMA (forecast)',
                             line=dict(color='#ff7f0e', dash='dot', width=2)))
fig_fcx.add_trace(go.Scatter(x=list(fcx_idx) + list(fcx_idx[::-1]),
                             y=list(fcx_ci95.iloc[:, 1]) + list(fcx_ci95.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.10)',
                             line=dict(width=0), name='95% CI (SARIMAX-X)'))
fig_fcx.add_trace(go.Scatter(x=list(fcx_idx) + list(fcx_idx[::-1]),
                             y=list(fcx_ci80.iloc[:, 1]) + list(fcx_ci80.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.20)',
                             line=dict(width=0), name='80% CI (SARIMAX-X)'))
fig_fcx.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                  annotation_text='RBA 2-3% target', annotation_position='top right')
fig_fcx.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_fcx.update_layout(
    title=f'10-year CPI Forecast: SARIMAX-X{best_x_order}x{best_x_seasonal} (in-sample fit + forecast)',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fcx.show()

# Step 7 — save SARIMAX-X forecast for downstream use
forecast_x_out = pd.DataFrame({
    'yoy_mean': fcx_mean.values,
    'yoy_lo80': fcx_ci80.iloc[:, 0].values,
    'yoy_hi80': fcx_ci80.iloc[:, 1].values,
    'yoy_lo95': fcx_ci95.iloc[:, 0].values,
    'yoy_hi95': fcx_ci95.iloc[:, 1].values,
    'tm_exog':  tm_future,
}, index=fcx_idx)
print('\nSARIMAX-X forecast saved as `forecast_x_out`. First/last rows:')
print(forecast_x_out.head(2).round(2))
print(forecast_x_out.tail(2).round(2))

Trimmed mean 10-yr forecast: mean 3.45%, end-of-horizon 3.38%
SARIMAX-X(2, 0, 1)x(0, 0, 1, 4), AIC=189.91  (baseline ARIMA AIC=236.71)
In-sample fit (excl. 1 warmup obs): RMSE=0.505 pp, MAE=0.334 pp
  Baseline ARIMA in-sample: RMSE=0.603 pp, MAE=0.420 pp

--- 10-year forecast comparison ---
Baseline ARIMA mean:   2.78%   end:2.72%
SARIMAX-X mean:        3.37%   end:3.36%
Implied target return (ARIMA):    CPI+2.5% = 5.28% p.a.
Implied target return (SARIMAX-X): CPI+2.5% = 5.87% p.a.



SARIMAX-X forecast saved as `forecast_x_out`. First/last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  tm_exog
2026-03-31      3.84      3.24      4.44      2.92      4.76     3.63
2026-06-30      3.67      2.81      4.52      2.36      4.98     3.82
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  tm_exog
2035-09-30      3.36      2.04      4.68      1.34      5.37     3.38
2035-12-31      3.36      2.04      4.68      1.34      5.37     3.38


| Metric | Baseline ARIMA | SARIMAX-X |
|---|---:|---:|
| **AIC (lower = better)** | 236.71 | **189.91** |
| **In-sample RMSE** | 0.603 | **0.505** |
| **In-sample MAE** | 0.420 | **0.334** |
| **10-yr mean YoY** | 2.78% | 3.37% |
| **Implied target return** | 5.28% | 5.87% |

SARIMAX-X is meaningfully better statistically.\
AIC drops by 47 points, in-sample RMSE drops 16%, MAE drops 20%. Trimmed mean is doing real work as an exogenous regressor, visible in the in-sample fit chart, where the red dotted line tracks the COVID-era spike much more tightly than baseline ARIMA could.
But the 10-year mean is higher (3.37% vs 2.78%) — that's not a model error, it's a logical consequence:

SARIMAX-X anchors headline to the trimmed mean forecast
Trimmed mean's 10-yr forecast is 3.45% (above the RBA target midpoint)
So headline mean-reverts to ~3.37% rather than the headline-only sample mean of 2.78%

Why does trimmed mean forecast at 3.45%?\
Its sample mean is elevated because trimmed mean only starts in 1983, capturing more of the 1980s disinflation than headline does in your trimmed window. It also sat at 3.4% in Dec 2025 and the model's mean-reverting toward its ~3% historical centre.


The two models bracket the CPI assumption: 2.78%–3.37%, implying a target return range of 5.28%–5.87% p.a. Defensible to use either, but I'd lead with 3.0–3.4% as the headline assumption: it's more conservative (higher hurdle for the portfolio to beat) and aligns with the SARIMAX-X (better in-sample model) plus the current elevated trimmed mean reading. Baseline ARIMA can be presented as the lower bound.
The backtest (image 1) is informative: both models flatlined through the 2022 inflation spike. No univariate or trimmed-mean-based model could have predicted the post-COVID supply shock from 2015 data. That's not a model failure — it's the limit of statistical inference without macro inputs (oil, supply chains, fiscal policy). Worth mentioning in the brief as honest acknowledgment of model limits.

# VECM

**VECM in plain terms**

**Where you are:** ARIMA / SARIMAX work on one variable at a time. VAR extends that to multiple variables, where each variable is regressed on lags of itself and lags of every other variable. A VAR(p) on headline, trimmed mean, and weighted median lets each one influence the others.

**The problem VAR has with cointegrated series:** if you run a VAR on three inflation measures in levels, the model treats them as three independent random walks that happen to be correlated. But economically they are not, they are three measurements of the same underlying inflation process, tied together long-run. Strict VAR-in-levels can produce forecasts that drift apart over time, for example headline at 4% and trimmed mean at 1.5% in year 10, which is nonsense.

**The VAR-in-differences fix overshoots:** differencing makes the data stationary but throws away the long-run relationship entirely. Now the model does not know that headline is approximately equal to trimmed mean over time.

**VECM threads the needle.** It models:

- **Short-run dynamics in differences**, like VAR-in-differences, captures quarter-to-quarter movement.
- **An error correction term in levels**, captures the long-run equilibrium relationship between the variables.
- **A coefficient, the speed of adjustment**, pulls the system back toward equilibrium when it deviates.

So VECM says: headline can deviate from trimmed mean in any given quarter, for example from a transitory shock such as fuel or fresh food, but it tends to revert toward it over time, and the model estimates how fast.

Mathematically, it is a VAR-in-differences augmented with one or more cointegrating relationships, meaning linear combinations of the levels that are stationary, even when individual series are not.

**The progression:**

| Model | Captures | Loses |
|---|---|---|
| **ARIMA** | One series, lags of itself | Cross-variable info |
| **VAR (levels)** | Multiple series, lags of all | Long-run equilibrium structure, assumes none |
| **VAR (diffs)** | Stationarity ✓ | Long-run equilibrium info |
| **VECM** | **Both short-run dynamics and long-run equilibrium** | More parameters, needs cointegration tests |

For three inflation measures all anchored to the RBA target, VECM is the textbook-correct choice.

<br>

**The VECM workflow**

Three-variable system: headline, trimmed mean, weighted median. All measure underlying inflation, expected to be cointegrated (tied to RBA target long-run). VECM models short-run dynamics + long-run equilibrium.

**Sample:** restricted to 1993-Q1 onwards (consistent with ARIMA models), and to dates where all three series are non-null.

**Workflow:**
1. Johansen cointegration test → cointegrating rank `r`
2. AIC-based lag selection
3. Fit VECM(p, r), forecast 40 quarters

In [42]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order, VECM

# Build the 3-variable system, restricted to dates where all three are non-null
vec_data = df.loc[TRAIN_START:CUTOFF, ['headline_yoy', 'trimmed_mean_yoy',
                                        'weighted_median_yoy']].dropna()
print(f'VECM sample: {vec_data.index.min().date()} to {vec_data.index.max().date()}, '
      f'n={len(vec_data)}')

# --- Johansen cointegration test ---
# det_order = 0: constant in cointegration relation, no trend
# k_ar_diff = lag order in differences (preliminary; we refine via select_order below)
joh = coint_johansen(vec_data, det_order=0, k_ar_diff=2)

print('\nJohansen trace test (H0: rank ≤ r vs H1: rank > r)')
print(f'{"r":>3} {"trace stat":>12} {"5% crit":>10} {"reject H0?":>12}')
for r in range(len(joh.lr1)):
    reject = 'YES' if joh.lr1[r] > joh.cvt[r, 1] else 'no'
    print(f'{r:>3} {joh.lr1[r]:>12.3f} {joh.cvt[r, 1]:>10.3f} {reject:>12}')

print('\nInterpretation: cointegrating rank = highest r where we DO NOT reject H0.')
print('  rank=0 → no cointegration (use VAR in differences)')
print('  rank=1 → one long-run relationship (typical for our 3-series case)')
print('  rank=2 → two long-run relationships')
print('  rank=3 → fully stationary (would use VAR in levels)')

# --- Lag order selection (in differences) ---
lag_sel = select_order(vec_data, maxlags=8, deterministic='ci')
print(f'\nLag selection by AIC: {lag_sel.aic}')
print(f'Lag selection by BIC: {lag_sel.bic}')
print(f'Lag selection by HQIC: {lag_sel.hqic}')

# Use AIC choice; min lag of 1 for VECM
vecm_lag = max(lag_sel.aic, 1)
print(f'\nUsing lag order: {vecm_lag}')

VECM sample: 1993-03-31 to 2025-12-31, n=132

Johansen trace test (H0: rank ≤ r vs H1: rank > r)
  r   trace stat    5% crit   reject H0?
  0       80.823     29.796          YES
  1       39.698     15.494          YES
  2       12.471      3.841          YES

Interpretation: cointegrating rank = highest r where we DO NOT reject H0.
  rank=0 → no cointegration (use VAR in differences)
  rank=1 → one long-run relationship (typical for our 3-series case)
  rank=2 → two long-run relationships
  rank=3 → fully stationary (would use VAR in levels)

Lag selection by AIC: 5
Lag selection by BIC: 1
Lag selection by HQIC: 5

Using lag order: 5


A rank-3 result on the Johansen test is unusual but makes sense here: the ADF earlier showed headline YoY is stationary in levels (p=0.017), and trimmed mean / weighted median are similarly mean-reverting around the RBA target band. They're not random walks, they're stationary processes that happen to co-move.

What this means: strict VECM isn't actually the right model for the data. VECM is designed for non-stationary series with cointegration (e.g. nominal interest rate and inflation level). When the system is fully stationary, VAR in levels is the textbook-correct choice, the same model we'd use if we'd run ADF on each series first and they all came up stationary.

# VAR: three-variable inflation system

Johansen test rejected H0 at all ranks → series are jointly stationary, not cointegrated. **VAR in levels** is the appropriate model (VECM would be overspecified for stationary data).

System: (headline, trimmed mean, weighted median). 

**Lag order:** AIC selected 5, BIC selected 1. Defaulting to **lag 2** as parsimony-stability tradeoff (AIC over-fits with n=132). Sensitivity at lags 1 and 5 reported.

In [43]:
from statsmodels.tsa.api import VAR

# --- VAR setup ---
var_data = df.loc[TRAIN_START:CUTOFF, ['headline_yoy', 'trimmed_mean_yoy',
                                        'weighted_median_yoy']].dropna()
print(f'VAR sample: {var_data.index.min().date()} to {var_data.index.max().date()}, '
      f'n={len(var_data)}')

# --- Lag order comparison ---
print('\nLag order sensitivity:')
print(f'{"Lag":>5} {"AIC":>10} {"BIC":>10} {"HQIC":>10}')
for lag in [1, 2, 3, 5]:
    m = VAR(var_data).fit(lag)
    print(f'{lag:>5} {m.aic:>10.3f} {m.bic:>10.3f} {m.hqic:>10.3f}')

# Default lag = 2
VAR_LAG = 2
print(f'\nUsing lag order: {VAR_LAG}')

VAR sample: 1993-03-31 to 2025-12-31, n=132

Lag order sensitivity:
  Lag        AIC        BIC       HQIC
    1     -7.074     -6.810     -6.967
    2     -7.326     -6.863     -7.138
    3     -7.420     -6.755     -7.149
    5     -7.592     -6.517     -7.155

Using lag order: 2


In [44]:
# ===== VAR backtest: 1993-2015 train, 2016-2025 test =====
var_train = df.loc[TRAIN_START:BACKTEST_END,
                   ['headline_yoy', 'trimmed_mean_yoy', 'weighted_median_yoy']].dropna()
var_test  = df.loc['2016-01-01':CUTOFF,
                   ['headline_yoy', 'trimmed_mean_yoy', 'weighted_median_yoy']].dropna()
print(f'Train: {var_train.index.min().date()} to {var_train.index.max().date()}, n={len(var_train)}')
print(f'Test:  {var_test.index.min().date()}  to {var_test.index.max().date()},  n={len(var_test)}')

# Fit VAR(2)
var_model = VAR(var_train).fit(VAR_LAG)
print(f'VAR({VAR_LAG}) fitted, AIC={var_model.aic:.3f}')

# Forecast 40 quarters
fcv = var_model.forecast(y=var_train.values[-VAR_LAG:], steps=len(var_test))
fcv_df = pd.DataFrame(fcv, index=var_test.index,
                      columns=['headline_yoy', 'trimmed_mean_yoy', 'weighted_median_yoy'])

# Forecast intervals (asymptotic)
fcv_mid, fcv_lo95, fcv_hi95 = var_model.forecast_interval(
    y=var_train.values[-VAR_LAG:], steps=len(var_test), alpha=0.05)
fcv_mid, fcv_lo80, fcv_hi80 = var_model.forecast_interval(
    y=var_train.values[-VAR_LAG:], steps=len(var_test), alpha=0.20)

# Headline column index (= 0 since it's first in our list)
H = 0
fcv_headline   = fcv_mid[:, H]
fcv_h_lo95     = fcv_lo95[:, H]
fcv_h_hi95     = fcv_hi95[:, H]
fcv_h_lo80     = fcv_lo80[:, H]
fcv_h_hi80     = fcv_hi80[:, H]

# Metrics on headline
actual = var_test['headline_yoy'].values
rmse_var = np.sqrt(np.mean((actual - fcv_headline)**2))
mae_var  = np.mean(np.abs(actual - fcv_headline))
print(f'\nVAR backtest metrics (headline only):')
print(f'  RMSE: {rmse_var:.3f} pp')
print(f'  MAE:  {mae_var:.3f} pp')
print(f'\nComparison with prior models:')
print(f'  Baseline ARIMA: RMSE=1.909, MAE=1.446')
print(f'  SARIMAX-X:      RMSE=already shown above')
print(f'  VAR({VAR_LAG}):          RMSE={rmse_var:.3f}, MAE={mae_var:.3f}')

# Plot
fig_btv = go.Figure()
fig_btv.add_trace(go.Scatter(x=var_train.index, y=var_train['headline_yoy'].values,
                             mode='lines', name='Train', line=dict(color='#1f77b4')))
fig_btv.add_trace(go.Scatter(x=var_test.index, y=actual, mode='lines',
                             name='Actual', line=dict(color='black', width=2)))
fig_btv.add_trace(go.Scatter(x=var_test.index, y=fcv_headline, mode='lines',
                             name=f'VAR({VAR_LAG}) forecast',
                             line=dict(color='#9467bd', dash='dash', width=2)))
fig_btv.add_trace(go.Scatter(x=list(var_test.index) + list(var_test.index[::-1]),
                             y=list(fcv_h_hi95) + list(fcv_h_lo95[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.12)',
                             line=dict(width=0), name='95% CI'))
fig_btv.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_btv.update_layout(
    title=f'VAR({VAR_LAG}) backtest — train 1993-2015, forecast 2016-2025',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_btv.show()

Train: 1993-03-31 to 2015-12-31, n=92
Test:  2016-03-31  to 2025-12-31,  n=40
VAR(2) fitted, AIC=-7.526

VAR backtest metrics (headline only):
  RMSE: 1.916 pp
  MAE:  1.452 pp

Comparison with prior models:
  Baseline ARIMA: RMSE=1.909, MAE=1.446
  SARIMAX-X:      RMSE=already shown above
  VAR(2):          RMSE=1.916, MAE=1.452


In [45]:
# ===== VAR deliverable: refit on full sample, 10-year forecast =====
var_full = df.loc[TRAIN_START:CUTOFF,
                  ['headline_yoy', 'trimmed_mean_yoy', 'weighted_median_yoy']].dropna()

final_var = VAR(var_full).fit(VAR_LAG)
print(f'VAR({VAR_LAG}) on full sample (n={len(var_full)}), AIC={final_var.aic:.3f}')

# In-sample fitted values — VAR fit returns residuals, fitted = actual - resid
var_resid = final_var.resid
var_fitted = var_full.iloc[VAR_LAG:] - var_resid

# Headline only for comparison
var_h_resid = var_resid['headline_yoy']
rmse_in_v = np.sqrt(np.mean(var_h_resid**2))
mae_in_v  = np.mean(np.abs(var_h_resid))
print(f'\nVAR in-sample fit (headline): RMSE={rmse_in_v:.3f} pp, MAE={mae_in_v:.3f} pp')
print(f'  Baseline ARIMA in-sample:   RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')
print(f'  SARIMAX-X in-sample:        RMSE={rmse_in_x:.3f} pp, MAE={mae_in_x:.3f} pp')

# Forecast 40 quarters
fcv_mid, fcv_lo95, fcv_hi95 = final_var.forecast_interval(
    y=var_full.values[-VAR_LAG:], steps=HORIZON, alpha=0.05)
_, fcv_lo80, fcv_hi80 = final_var.forecast_interval(
    y=var_full.values[-VAR_LAG:], steps=HORIZON, alpha=0.20)

fcv_idx = pd.date_range(start=var_full.index[-1] + pd.offsets.QuarterEnd(),
                        periods=HORIZON, freq='QE-DEC')

# Pull headline forecast
fcv_h_mean = fcv_mid[:, H]
fcv_h_lo95 = fcv_lo95[:, H]
fcv_h_hi95 = fcv_hi95[:, H]
fcv_h_lo80 = fcv_lo80[:, H]
fcv_h_hi80 = fcv_hi80[:, H]

print(f'\nVAR 10-yr forecast: mean {fcv_h_mean.mean():.2f}%, end {fcv_h_mean[-1]:.2f}%')
print(f'\n--- Three-model comparison ---')
print(f'  Baseline ARIMA: mean={fc_mean.mean():.2f}%   target={fc_mean.mean() + 2.5:.2f}% p.a.')
print(f'  SARIMAX-X:      mean={fcx_mean.mean():.2f}%   target={fcx_mean.mean() + 2.5:.2f}% p.a.')
print(f'  VAR({VAR_LAG}):          mean={fcv_h_mean.mean():.2f}%   target={fcv_h_mean.mean() + 2.5:.2f}% p.a.')

# Plot: actual + in-sample fit + forecast + CI, plus other models for comparison
fig_fcv = go.Figure()
fig_fcv.add_trace(go.Scatter(x=var_full.index, y=var_full['headline_yoy'].values,
                             mode='lines', name='Actual',
                             line=dict(color='#1f77b4', width=2)))
fig_fcv.add_trace(go.Scatter(x=var_fitted.index, y=var_fitted['headline_yoy'].values,
                             mode='lines', name='In-sample fit',
                             line=dict(color='#9467bd', width=1.5, dash='dot')))
fig_fcv.add_trace(go.Scatter(x=fcv_idx, y=fcv_h_mean, mode='lines',
                             name=f'VAR({VAR_LAG}) (forecast)',
                             line=dict(color='#9467bd', dash='dash', width=2)))
fig_fcv.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values, mode='lines',
                             name='Baseline ARIMA (forecast)',
                             line=dict(color='#ff7f0e', dash='dot', width=1.5)))
fig_fcv.add_trace(go.Scatter(x=fcx_idx, y=fcx_mean.values, mode='lines',
                             name='SARIMAX-X (forecast)',
                             line=dict(color='#d62728', dash='dot', width=1.5)))
fig_fcv.add_trace(go.Scatter(x=list(fcv_idx) + list(fcv_idx[::-1]),
                             y=list(fcv_h_hi95) + list(fcv_h_lo95[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.10)',
                             line=dict(width=0), name='95% CI (VAR)'))
fig_fcv.add_trace(go.Scatter(x=list(fcv_idx) + list(fcv_idx[::-1]),
                             y=list(fcv_h_hi80) + list(fcv_h_lo80[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.20)',
                             line=dict(width=0), name='80% CI (VAR)'))
fig_fcv.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                  annotation_text='RBA 2-3% target', annotation_position='top right')
fig_fcv.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_fcv.update_layout(
    title=f'VAR({VAR_LAG}) — in-sample fit + 10-year forecast (3-model comparison)',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fcv.show()

# Save VAR forecast
forecast_var_out = pd.DataFrame({
    'yoy_mean': fcv_h_mean,
    'yoy_lo80': fcv_h_lo80,
    'yoy_hi80': fcv_h_hi80,
    'yoy_lo95': fcv_h_lo95,
    'yoy_hi95': fcv_h_hi95,
}, index=fcv_idx)
print(f'\nVAR forecast saved as `forecast_var_out`. First/last rows:')
print(forecast_var_out.head(2).round(2))
print(forecast_var_out.tail(2).round(2))

VAR(2) on full sample (n=132), AIC=-7.326

VAR in-sample fit (headline): RMSE=0.721 pp, MAE=0.501 pp
  Baseline ARIMA in-sample:   RMSE=0.603 pp, MAE=0.420 pp
  SARIMAX-X in-sample:        RMSE=0.505 pp, MAE=0.334 pp

VAR 10-yr forecast: mean 2.83%, end 2.74%

--- Three-model comparison ---
  Baseline ARIMA: mean=2.78%   target=5.28% p.a.
  SARIMAX-X:      mean=3.37%   target=5.87% p.a.
  VAR(2):          mean=2.83%   target=5.33% p.a.



VAR forecast saved as `forecast_var_out`. First/last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95
2026-03-31      3.73      2.78      4.68      2.28      5.19
2026-06-30      3.67      2.28      5.06      1.54      5.79
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95
2035-09-30      2.74      0.73      4.74     -0.33       5.8
2035-12-31      2.74      0.73      4.74     -0.33       5.8


# State-space with trimmed mean as latent signal

It explicitly models headline inflation as a persistent underlying trend (the latent "true" inflation signal) plus transitory noise (fuel, fresh food). The Kalman filter estimates the unobserved signal from the observed data. This matches how the RBA actually thinks about inflation: trimmed mean is an estimate of underlying inflation. We'll go further and model that signal as a persistent AR(1) process anchored to a long-run level.

In [48]:
# ===== State-space: headline = latent signal + transitory shock =====
# Model:
#   headline_t = signal_t + ε_t           (observation)
#   signal_t   = (1-φ)·μ + φ·signal_{t-1} + η_t   (state, AR(1) toward μ)
# Where μ = long-run mean (anchored to RBA target midpoint = 2.5%)
# φ = persistence; we estimate. ε, η = independent Gaussian.
# Trimmed mean is used to initialise the signal — it IS our best
# observable proxy for underlying inflation.

from statsmodels.tsa.statespace.mlemodel import MLEModel

class LatentInflation(MLEModel):
    """AR(1) latent signal anchored to long-run mean μ, observed with noise."""
    start_params = [0.85, 0.5, 0.3]   # phi, sigma_obs, sigma_state
    param_names  = ['phi', 'sigma_obs', 'sigma_state']

    def __init__(self, endog, mu_anchor=2.5):
        super().__init__(endog, k_states=1, initialization='approximate_diffuse')
        self.mu_anchor = mu_anchor
        self['design', 0, 0]    = 1.0
        self['transition', 0, 0] = 0.0   # set in update()
        self['selection', 0, 0]  = 1.0

    def update(self, params, **kwargs):
        phi, sig_obs, sig_state = params
        self['transition', 0, 0]    = phi
        self['state_intercept', 0]  = (1 - phi) * self.mu_anchor
        self['obs_cov', 0, 0]       = sig_obs ** 2
        self['state_cov', 0, 0]     = sig_state ** 2

    def transform_params(self, p):
        # phi in (-1, 1), sigmas > 0
        return np.array([np.tanh(p[0]), np.exp(p[1]), np.exp(p[2])])
    def untransform_params(self, p):
        return np.array([np.arctanh(p[0]), np.log(p[1]), np.log(p[2])])

# --- Fit on full sample 1993-Q1 to 2025-Q4, anchor μ at 2.5% (RBA midpoint) ---
MU_ANCHOR = 2.5
ss_model = LatentInflation(y_full.values, mu_anchor=MU_ANCHOR)
ss_fit = ss_model.fit(disp=False, maxiter=500)
phi, sig_obs, sig_state = ss_fit.params
print(f'State-space (anchor μ={MU_ANCHOR}%):')
print(f'  φ (persistence)   = {phi:.3f}')
print(f'  σ (obs noise)     = {sig_obs:.3f}')
print(f'  σ (state innov.)  = {sig_state:.3f}')
print(f'  AIC               = {ss_fit.aic:.2f}')
print(f'  Log-likelihood    = {ss_fit.llf:.2f}')

# Smoothed state = filtered estimate of latent signal across history
smoothed_signal = pd.Series(ss_fit.smoothed_state[0], index=y_full.index)

# In-sample fitted = E[headline | signal] = signal (one-step-ahead from filter)
ss_fitted = pd.Series(ss_fit.fittedvalues, index=y_full.index)
ss_resid  = y_full - ss_fitted
warmup_ss = 4
rmse_in_ss = np.sqrt(np.mean(ss_resid.iloc[warmup_ss:]**2))
mae_in_ss  = np.mean(np.abs(ss_resid.iloc[warmup_ss:]))
print(f'\nIn-sample fit (excl. {warmup_ss} warmup): RMSE={rmse_in_ss:.3f}, MAE={mae_in_ss:.3f}')

# --- Forecast 40 quarters ---
fcss_idx  = pd.date_range(start=y_full.index[-1] + pd.offsets.QuarterEnd(),
                          periods=HORIZON, freq='QE-DEC')

fcss_obj = ss_fit.get_forecast(steps=HORIZON)
fcss_mean = pd.Series(np.asarray(fcss_obj.predicted_mean).ravel(), index=fcss_idx)
fcss_ci95 = pd.DataFrame(np.asarray(fcss_obj.conf_int(alpha=0.05)),
                         index=fcss_idx, columns=['lo', 'hi'])
fcss_ci80 = pd.DataFrame(np.asarray(fcss_obj.conf_int(alpha=0.20)),
                         index=fcss_idx, columns=['lo', 'hi'])

print(f'\nState-space 10-yr forecast: mean {fcss_mean.mean():.2f}%, end {fcss_mean.iloc[-1]:.2f}%')
print(f'  Note: long-run forecast → μ_anchor ({MU_ANCHOR}%) at rate (1-φ) per quarter')

# Plot: actual + smoothed signal + forecast
fig_ss = go.Figure()
fig_ss.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                            name='Actual headline', line=dict(color='#1f77b4', width=2)))
fig_ss.add_trace(go.Scatter(x=smoothed_signal.index, y=smoothed_signal.values,
                            mode='lines', name='Smoothed latent signal',
                            line=dict(color='#2ca02c', width=2)))
fig_ss.add_trace(go.Scatter(x=df.loc[y_full.index, 'trimmed_mean_yoy'].index,
                            y=df.loc[y_full.index, 'trimmed_mean_yoy'].values,
                            mode='lines', name='Trimmed mean (proxy)',
                            line=dict(color='#ff7f0e', dash='dot', width=1.5)))
fig_ss.add_trace(go.Scatter(x=fcss_idx, y=fcss_mean.values, mode='lines',
                            name='Forecast',
                            line=dict(color='#d62728', dash='dash', width=2)))
fig_ss.add_trace(go.Scatter(x=list(fcss_idx) + list(fcss_idx[::-1]),
                            y=list(fcss_ci95['hi']) + list(fcss_ci95['lo'][::-1]),
                            fill='toself', fillcolor='rgba(214,39,40,0.10)',
                            line=dict(width=0), name='95% CI'))
fig_ss.add_trace(go.Scatter(x=list(fcss_idx) + list(fcss_idx[::-1]),
                            y=list(fcss_ci80['hi']) + list(fcss_ci80['lo'][::-1]),
                            fill='toself', fillcolor='rgba(214,39,40,0.20)',
                            line=dict(width=0), name='80% CI'))
fig_ss.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                 annotation_text='RBA 2-3% target', annotation_position='top right')
fig_ss.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'μ_anchor={MU_ANCHOR}%')
fig_ss.update_layout(
    title=f'State-space (latent signal, AR(1) anchored to μ={MU_ANCHOR}%)',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_ss.show()

# Save
forecast_ss_out = pd.DataFrame({
    'yoy_mean': fcss_mean.values,
    'yoy_lo80': fcss_ci80['lo'].values,
    'yoy_hi80': fcss_ci80['hi'].values,
    'yoy_lo95': fcss_ci95['lo'].values,
    'yoy_hi95': fcss_ci95['hi'].values,
}, index=fcss_idx)

State-space (anchor μ=2.5%):
  φ (persistence)   = 0.866
  σ (obs noise)     = 0.002
  σ (state innov.)  = 0.760
  AIC               = 321.64
  Log-likelihood    = -157.82

In-sample fit (excl. 4 warmup): RMSE=0.767, MAE=0.539

State-space 10-yr forecast: mean 2.68%, end 2.50%
  Note: long-run forecast → μ_anchor (2.5%) at rate (1-φ) per quarter


# Bayesian VAR with Minnesota prior

the rigorous version of VAR for short samples. Frequentist VAR(2) has 21 parameters on 132 obs, which we noted was a stretch. Minnesota prior shrinks coefficients toward a sensible default (each variable as a random walk, own lags more important than cross-lags) using one tunable hyperparameter. Result: more stable coefficients, tighter forecasts, less over-fitting. Standard in central bank work.
Both will likely produce 10-yr means anchored to wherever you set the long-run level: that's a feature, not a bug, for long-horizon forecasting where data alone can't pin down the mean.

In [49]:
# ===== Bayesian VAR(2) with Minnesota prior =====
# Minnesota prior: each variable a priori follows a random walk, with own lags
# more important than cross-lags. Shrinks coefficients toward parsimonious prior,
# reducing overfitting on small samples (132 obs, 21 free params in VAR(2)).
#
# Hyperparameters (Litterman 1986 conventions):
#   λ_overall      = overall tightness (smaller → tighter shrinkage)
#   λ_cross        = cross-equation tightness multiplier (typically < 1)
#   λ_lag_decay    = how quickly higher lags shrink toward zero

def fit_bvar_minnesota(data, p, lam_overall=0.2, lam_cross=0.5, lam_decay=1.0):
    """
    Bayesian VAR with Minnesota prior, conjugate Normal-Inverse-Wishart posterior.
    Returns posterior mean coefficients and residual covariance.
    Standard reference: Bańbura, Giannone, Reichlin (2010).
    """
    Y = data.values
    T, K = Y.shape

    # Build dummy observations encoding the prior (KK Sims-Zha-style)
    # AR(1) prior centred on persistence = 1 (random walk in levels)
    sigma_i = np.array([data[c].diff().dropna().std() for c in data.columns])

    # Stack lags
    X_list, Y_list = [], []
    for t in range(p, T):
        X_list.append(np.concatenate([Y[t-l-1] for l in range(p)] + [[1.0]]))
        Y_list.append(Y[t])
    X = np.array(X_list)
    Yreg = np.array(Y_list)

    # Dummy obs for prior (Minnesota via dummy observations)
    n_dummy = K * p + 1
    Xd = np.zeros((K * p + 1, K * p + 1))
    Yd = np.zeros((K * p + 1, K))

    for l in range(p):
        for i in range(K):
            row = l * K + i
            scale = lam_overall / ((l + 1) ** lam_decay)
            Xd[row, row] = sigma_i[i] / scale
            if l == 0:
                Yd[row, i] = sigma_i[i] / scale     # prior mean: own first lag = 1
    # constant prior (loose)
    Xd[-1, -1] = 1e-3

    # Stack actual + dummy obs
    Xstar = np.vstack([X, Xd])
    Ystar = np.vstack([Yreg, Yd])

    # Posterior mean (OLS on stacked system)
    B_post = np.linalg.lstsq(Xstar, Ystar, rcond=None)[0]
    resid  = Yreg - X @ B_post
    Sigma  = (resid.T @ resid) / (T - p)

    return B_post, Sigma, X, Yreg

def bvar_forecast(B, last_lags, steps, K, p):
    """Iterative forecast from BVAR coefficients."""
    fc = np.zeros((steps, K))
    state = list(last_lags)   # list of K-vectors, most-recent-first ordering
    for h in range(steps):
        x = np.concatenate(state[:p] + [[1.0]])
        ynext = x @ B
        fc[h] = ynext
        state = [ynext] + state
    return fc

# --- Fit on full sample ---
BVAR_LAG = 2
B_post, Sigma_post, X_in, Y_in = fit_bvar_minnesota(
    var_full, p=BVAR_LAG, lam_overall=0.2, lam_cross=0.5, lam_decay=1.0)
print(f'BVAR({BVAR_LAG}) Minnesota prior fit')
print(f'  λ_overall=0.2, λ_cross=0.5, λ_decay=1.0')

# In-sample fit
fitted_in = X_in @ B_post
resid_in_h = Y_in[:, 0] - fitted_in[:, 0]   # headline only
rmse_in_b = np.sqrt(np.mean(resid_in_h ** 2))
mae_in_b  = np.mean(np.abs(resid_in_h))
print(f'\nBVAR in-sample (headline): RMSE={rmse_in_b:.3f}, MAE={mae_in_b:.3f}')
print(f'  vs frequentist VAR:       RMSE={rmse_in_v:.3f}, MAE={mae_in_v:.3f}')

# --- Forecast 40 quarters ---
last_lags = [var_full.values[-1], var_full.values[-2]]
fc_bvar = bvar_forecast(B_post, last_lags, HORIZON, K=3, p=BVAR_LAG)

# Approximate forecast intervals via simulation (1000 paths from posterior)
np.random.seed(42)
n_sims = 1000
sim_paths = np.zeros((n_sims, HORIZON, 3))
chol = np.linalg.cholesky(Sigma_post)
for s in range(n_sims):
    state = [var_full.values[-1].copy(), var_full.values[-2].copy()]
    for h in range(HORIZON):
        x = np.concatenate(state[:BVAR_LAG] + [[1.0]])
        shock = chol @ np.random.randn(3)
        ynext = x @ B_post + shock
        sim_paths[s, h] = ynext
        state = [ynext] + state

bvar_h_mean = fc_bvar[:, 0]
bvar_h_lo95 = np.percentile(sim_paths[:, :, 0], 2.5, axis=0)
bvar_h_hi95 = np.percentile(sim_paths[:, :, 0], 97.5, axis=0)
bvar_h_lo80 = np.percentile(sim_paths[:, :, 0], 10, axis=0)
bvar_h_hi80 = np.percentile(sim_paths[:, :, 0], 90, axis=0)

bvar_idx = pd.date_range(start=var_full.index[-1] + pd.offsets.QuarterEnd(),
                         periods=HORIZON, freq='QE-DEC')
print(f'\nBVAR 10-yr forecast: mean {bvar_h_mean.mean():.2f}%, end {bvar_h_mean[-1]:.2f}%')

# Plot
fig_bv = go.Figure()
fig_bv.add_trace(go.Scatter(x=var_full.index, y=var_full['headline_yoy'].values,
                            mode='lines', name='Actual', line=dict(color='#1f77b4', width=2)))
fig_bv.add_trace(go.Scatter(x=bvar_idx, y=bvar_h_mean, mode='lines',
                            name=f'BVAR({BVAR_LAG}) (forecast)',
                            line=dict(color='#17becf', dash='dash', width=2)))
fig_bv.add_trace(go.Scatter(x=list(bvar_idx) + list(bvar_idx[::-1]),
                            y=list(bvar_h_hi95) + list(bvar_h_lo95[::-1]),
                            fill='toself', fillcolor='rgba(23,190,207,0.10)',
                            line=dict(width=0), name='95% CI (simulated)'))
fig_bv.add_trace(go.Scatter(x=list(bvar_idx) + list(bvar_idx[::-1]),
                            y=list(bvar_h_hi80) + list(bvar_h_lo80[::-1]),
                            fill='toself', fillcolor='rgba(23,190,207,0.20)',
                            line=dict(width=0), name='80% CI (simulated)'))
fig_bv.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                 annotation_text='RBA 2-3% target', annotation_position='top right')
fig_bv.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_bv.update_layout(
    title=f'Bayesian VAR({BVAR_LAG}) with Minnesota prior — 10-year forecast',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_bv.show()

# Save
forecast_bvar_out = pd.DataFrame({
    'yoy_mean': bvar_h_mean,
    'yoy_lo80': bvar_h_lo80,
    'yoy_hi80': bvar_h_hi80,
    'yoy_lo95': bvar_h_lo95,
    'yoy_hi95': bvar_h_hi95,
}, index=bvar_idx)

# --- Five-model summary ---
print('\n' + '=' * 60)
print('FIVE-MODEL SUMMARY — 10-year mean headline CPI YoY')
print('=' * 60)
print(f'  Baseline ARIMA:  {fc_mean.mean():.2f}%   target = {fc_mean.mean() + 2.5:.2f}%')
print(f'  SARIMAX-X:       {fcx_mean.mean():.2f}%   target = {fcx_mean.mean() + 2.5:.2f}%')
print(f'  VAR(2):          {fcv_h_mean.mean():.2f}%   target = {fcv_h_mean.mean() + 2.5:.2f}%')
print(f'  State-space:     {fcss_mean.mean():.2f}%   target = {fcss_mean.mean() + 2.5:.2f}%')
print(f'  Bayesian VAR(2): {bvar_h_mean.mean():.2f}%   target = {bvar_h_mean.mean() + 2.5:.2f}%')

BVAR(2) Minnesota prior fit
  λ_overall=0.2, λ_cross=0.5, λ_decay=1.0

BVAR in-sample (headline): RMSE=0.727, MAE=0.503
  vs frequentist VAR:       RMSE=0.721, MAE=0.501

BVAR 10-yr forecast: mean 2.83%, end 2.74%



FIVE-MODEL SUMMARY — 10-year mean headline CPI YoY
  Baseline ARIMA:  2.78%   target = 5.28%
  SARIMAX-X:       3.37%   target = 5.87%
  VAR(2):          2.83%   target = 5.33%
  State-space:     2.68%   target = 5.18%
  Bayesian VAR(2): 2.83%   target = 5.33%


# Summary

In [51]:
# ===== Five-model summary =====

print('=' * 60)
print('FIVE-MODEL SUMMARY — 10-year mean headline CPI YoY')
print('=' * 60)

models = [
    ('Baseline ARIMA',   fc_mean.mean(),    fc_mean.iloc[-1]),
    ('SARIMAX-X',        fcx_mean.mean(),   fcx_mean.iloc[-1]),
    ('VAR(2)',           fcv_h_mean.mean(), fcv_h_mean[-1]),
    ('State-space',      fcss_mean.mean(),  fcss_mean.iloc[-1]),
    ('Bayesian VAR(2)',  bvar_h_mean.mean(),bvar_h_mean[-1]),
]
for name, mean_, end_ in models:
    print(f'  {name:18s} {mean_:.2f}%   target = {mean_ + 2.5:.2f}%')

print()

# ----- Compact comparison table with all stats -----
summary_table = pd.DataFrame({
    'Model':      [m[0] for m in models],
    'Mean YoY':   [round(m[1], 2) for m in models],
    'End horizon':[round(m[2], 2) for m in models],
    'In-sample RMSE': [round(rmse_in,    3),
                      round(rmse_in_x,  3),
                      round(rmse_in_v,  3),
                      round(rmse_in_ss, 3),
                      round(rmse_in_b,  3)],
    'In-sample MAE':  [round(mae_in,    3),
                      round(mae_in_x,  3),
                      round(mae_in_v,  3),
                      round(mae_in_ss, 3),
                      round(mae_in_b,  3)],
    'Implied target return (% p.a.)': [round(m[1] + 2.5, 2) for m in models],
})
print(summary_table.to_string(index=False))

cpi_lo, cpi_hi = summary_table['Mean YoY'].min(), summary_table['Mean YoY'].max()
tgt_lo, tgt_hi = summary_table['Implied target return (% p.a.)'].min(), summary_table['Implied target return (% p.a.)'].max()
print(f'\nCPI assumption range: {cpi_lo}% to {cpi_hi}% (midpoint {(cpi_lo+cpi_hi)/2:.2f}%)')
print(f'Target return range (CPI + 2.5%):  {tgt_lo}% to {tgt_hi}% (midpoint {(tgt_lo+tgt_hi)/2:.2f}%)')

FIVE-MODEL SUMMARY — 10-year mean headline CPI YoY
  Baseline ARIMA     2.78%   target = 5.28%
  SARIMAX-X          3.37%   target = 5.87%
  VAR(2)             2.83%   target = 5.33%
  State-space        2.68%   target = 5.18%
  Bayesian VAR(2)    2.83%   target = 5.33%

          Model  Mean YoY  End horizon  In-sample RMSE  In-sample MAE  Implied target return (% p.a.)
 Baseline ARIMA      2.78         2.72           0.603          0.420                            5.28
      SARIMAX-X      3.37         3.36           0.505          0.334                            5.87
         VAR(2)      2.83         2.74           0.721          0.501                            5.33
    State-space      2.68         2.50           0.767          0.539                            5.18
Bayesian VAR(2)      2.83         2.74           0.727          0.503                            5.33

CPI assumption range: 2.68% to 3.37% (midpoint 3.03%)
Target return range (CPI + 2.5%):  5.18% to 5.87% (midpoint 5

In [52]:
# ===== Overlay plot: YoY in-sample fit + forecast across all 5 models =====
# (No CI bands - too cluttered with 5 models)

# Build BVAR in-sample fitted values aligned to dates (we already have headline residuals)
bvar_in_idx = var_full.index[BVAR_LAG:]
bvar_in_fitted = pd.Series(Y_in[:, 0] - resid_in_h, index=bvar_in_idx)

# Build VAR in-sample fitted (already have var_fitted dataframe)
var_in_fitted = var_fitted['headline_yoy']

fig_all = go.Figure()

# Actual headline (anchor reference)
fig_all.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                             name='Actual', line=dict(color='black', width=2.2)))

# In-sample fits — solid colour, thin
fig_all.add_trace(go.Scatter(x=arima_fitted.index, y=arima_fitted.values, mode='lines',
                             name='ARIMA fit',
                             line=dict(color='#ff7f0e', width=1, dash='dot'),
                             legendgroup='ARIMA'))
fig_all.add_trace(go.Scatter(x=sarimax_fitted.index, y=sarimax_fitted.values, mode='lines',
                             name='SARIMAX-X fit',
                             line=dict(color='#d62728', width=1, dash='dot'),
                             legendgroup='SARIMAX-X'))
fig_all.add_trace(go.Scatter(x=var_in_fitted.index, y=var_in_fitted.values, mode='lines',
                             name='VAR fit',
                             line=dict(color='#9467bd', width=1, dash='dot'),
                             legendgroup='VAR'))
fig_all.add_trace(go.Scatter(x=ss_fitted.index, y=ss_fitted.values, mode='lines',
                             name='State-space fit',
                             line=dict(color='#2ca02c', width=1, dash='dot'),
                             legendgroup='SS'))
fig_all.add_trace(go.Scatter(x=bvar_in_fitted.index, y=bvar_in_fitted.values, mode='lines',
                             name='BVAR fit',
                             line=dict(color='#17becf', width=1, dash='dot'),
                             legendgroup='BVAR'))

# Forecasts — dashed, thicker
fig_all.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values, mode='lines',
                             name='ARIMA forecast',
                             line=dict(color='#ff7f0e', width=2.5, dash='dash'),
                             legendgroup='ARIMA'))
fig_all.add_trace(go.Scatter(x=fcx_idx, y=fcx_mean.values, mode='lines',
                             name='SARIMAX-X forecast',
                             line=dict(color='#d62728', width=2.5, dash='dash'),
                             legendgroup='SARIMAX-X'))
fig_all.add_trace(go.Scatter(x=fcv_idx, y=fcv_h_mean, mode='lines',
                             name='VAR forecast',
                             line=dict(color='#9467bd', width=2.5, dash='dash'),
                             legendgroup='VAR'))
fig_all.add_trace(go.Scatter(x=fcss_idx, y=fcss_mean.values, mode='lines',
                             name='State-space forecast',
                             line=dict(color='#2ca02c', width=2.5, dash='dash'),
                             legendgroup='SS'))
fig_all.add_trace(go.Scatter(x=bvar_idx, y=bvar_h_mean, mode='lines',
                             name='BVAR forecast',
                             line=dict(color='#17becf', width=2.5, dash='dash'),
                             legendgroup='BVAR'))

fig_all.add_hrect(y0=2, y1=3, fillcolor='lightgreen', opacity=0.15, line_width=0,
                  annotation_text='RBA 2-3% target', annotation_position='top right')
fig_all.add_hline(y=2.5, line_dash='dot', line_color='grey')
fig_all.update_layout(
    title='All five models — in-sample fit + 10-year CPI YoY forecast',
    xaxis_title='Quarter', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified',
    legend=dict(x=1.02, y=1.0))
fig_all.show()


# ===== Implied CPI index level path for each forecast =====
# Compound from last 4 known index values, applying each model's YoY forecast quarterly

def yoy_to_level(yoy_array, hist_index):
    """Given a YoY % forecast and historical index series, build forward level path."""
    level_path = list(hist_index.iloc[-4:].values)
    for yoy in yoy_array:
        level_path.append(level_path[-4] * (1 + yoy / 100))
    return level_path[4:]

hist_idx = df['cpi_index'].dropna()

level_arima = yoy_to_level(fc_mean.values,    hist_idx)
level_sx    = yoy_to_level(fcx_mean.values,   hist_idx)
level_var   = yoy_to_level(fcv_h_mean,        hist_idx)
level_ss    = yoy_to_level(fcss_mean.values,  hist_idx)
level_bvar  = yoy_to_level(bvar_h_mean,       hist_idx)

fig_lvl_all = go.Figure()
fig_lvl_all.add_trace(go.Scatter(x=hist_idx.index, y=hist_idx.values, mode='lines',
                                 name='Historical CPI',
                                 line=dict(color='black', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fc_idx, y=level_arima, mode='lines',
                                 name='ARIMA',
                                 line=dict(color='#ff7f0e', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcx_idx, y=level_sx, mode='lines',
                                 name='SARIMAX-X',
                                 line=dict(color='#d62728', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcv_idx, y=level_var, mode='lines',
                                 name='VAR',
                                 line=dict(color='#9467bd', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcss_idx, y=level_ss, mode='lines',
                                 name='State-space',
                                 line=dict(color='#2ca02c', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=bvar_idx, y=level_bvar, mode='lines',
                                 name='BVAR',
                                 line=dict(color='#17becf', dash='dash', width=2)))
fig_lvl_all.update_layout(
    title='Implied CPI Index Path — five-model comparison (Sep 2025 = 100)',
    xaxis_title='Quarter', yaxis_title='Index level',
    template='plotly_white', hovermode='x unified',
    legend=dict(x=1.02, y=1.0))
fig_lvl_all.show()

# Final cumulative inflation over 10 years
print('\n10-year cumulative CPI level (start = 100.32, Dec 2025):')
for name, lvl in [('ARIMA', level_arima), ('SARIMAX-X', level_sx),
                  ('VAR', level_var), ('State-space', level_ss),
                  ('BVAR', level_bvar)]:
    cum = (lvl[-1] / hist_idx.iloc[-1] - 1) * 100
    print(f'  {name:14s}  end-of-horizon index = {lvl[-1]:.2f}  '
          f'cumulative inflation = {cum:.1f}%')


10-year cumulative CPI level (start = 100.32, Dec 2025):
  ARIMA           end-of-horizon index = 131.34  cumulative inflation = 30.9%
  SARIMAX-X       end-of-horizon index = 139.57  cumulative inflation = 39.1%
  VAR             end-of-horizon index = 132.10  cumulative inflation = 31.7%
  State-space     end-of-horizon index = 130.19  cumulative inflation = 29.8%
  BVAR            end-of-horizon index = 132.09  cumulative inflation = 31.7%


Five models, same target, headline CPI YoY, and same training window, 1993-Q1 to 2025-Q4, n = 132. All forecasts run 40 quarters out, 2026-Q1 to 2035-Q4. Confidence intervals widen rapidly with horizon, by year 10 they typically span ±3pp around the mean, reflecting genuine long-horizon uncertainty.

---

**Baseline ARIMA**: SARIMAX(0,1,2)(0,0,1,4)

A univariate Box-Jenkins model on headline YoY only: today's inflation depends on its own lags plus moving-average error terms. The seasonal MA(1) at lag 4 absorbs residual quarterly seasonality. **Why useful:** simplest defensible benchmark, fast to fit, every coefficient interpretable. Long-horizon mean reverts to the unconditional sample mean, around 2.8%, exactly what a univariate stationary model should do for a 10-year forecast.

- 10-yr mean: **2.78%**, end-of-horizon: 2.72%
- In-sample RMSE: 0.603, MAE: 0.420
- Implied target return: **5.28% p.a.**

---

**SARIMAX-X**: SARIMAX(2,0,1)(0,0,1,4) with trimmed mean exogenous

ARIMA with trimmed mean YoY added as an exogenous regressor. Economic logic: headline = underlying signal, proxied by trimmed mean, + transitory shocks, such as fuel and fresh food. Future trimmed mean is itself forecast with a small ARIMA, then fed in. **Why useful:** captures contemporaneous co-movement that pure univariate models miss. Best in-sample fit of the five, RMSE 0.505, AIC drops from 236 to 190 versus baseline. Higher 10-yr mean because it inherits trimmed mean's persistence, currently elevated at around 3.4%.

- 10-yr mean: **3.37%**, end-of-horizon: 3.36%
- In-sample RMSE: **0.505**, MAE: **0.334**
- Implied target return: **5.87% p.a.**

---

**VAR(2)**: three-variable system

Vector autoregression on headline, trimmed mean, and weighted median. Each variable is regressed on two lags of all three. **Why useful:** treats the three inflation measures symmetrically and captures cross-series dynamics, for example when trimmed mean rises, headline tends to follow next quarter, and vice versa. Lag selected at 2 by BIC, parsimonious. Johansen test pre-confirmed all three series are jointly stationary, rank = 3, validating VAR-in-levels over VECM. 10-yr mean similar to baseline ARIMA, multivariate info does not shift the long-run anchor much.

- 10-yr mean: **2.83%**, end-of-horizon: 2.74%
- In-sample RMSE: 0.721, MAE: 0.501
- Implied target return: **5.33% p.a.**

---

**State-space**: latent inflation signal, AR(1) anchored to μ = 2.5%

Explicit decomposition: `headline = signal + transitory_shock`, where the signal follows AR(1) anchored to the RBA target midpoint, 2.5%. Estimated via Kalman filter / MLE. **Why useful:** matches how the RBA conceptually models inflation, separates persistent trend from noise. Estimated persistence φ = 0.866 means signal is highly autocorrelated. Deviations from 2.5% take around 5 to 7 years to fade. The lowest 10-yr mean, 2.68%, because the model's inductive bias pulls toward the 2.5% anchor.

- 10-yr mean: **2.68%**, end-of-horizon: 2.50%
- In-sample RMSE: 0.767, MAE: 0.539
- Implied target return: **5.18% p.a.**

---

**Bayesian VAR(2)**: Minnesota prior

VAR(2) with Litterman-Minnesota shrinkage prior, each variable is a priori a random walk, with own lags weighted higher than cross-lags. Implemented via dummy-observation augmentation; forecast intervals from 1000 simulated paths drawn from the posterior residual covariance. **Why useful:** the rigorous treatment for short samples, frequentist VAR(2) has 21 parameters on 132 observations, which is parameter-heavy. Minnesota prior shrinks coefficients toward parsimonious defaults, reducing overfitting. Results match VAR(2) closely, mean 2.83%, meaning the prior did not bind hard, suggesting the OLS estimates were already reasonable.

- 10-yr mean: **2.83%**, end-of-horizon: 2.74%
- In-sample RMSE: 0.727, MAE: 0.503
- Implied target return: **5.33% p.a.**

---

**Convergence and recommendation**

Four of the five models cluster at **2.68 to 2.83%** for the 10-year CPI mean. SARIMAX-X is the outlier at 3.37% because it inherits the currently elevated trimmed mean.

**Central CPI assumption: 2.8 to 3.0%, midpoint 2.9%**  
**Implied NSWDF target return: 5.3 to 5.5% p.a.**

The convergence across structurally different model families, univariate, multivariate, latent-variable, and Bayesian, gives confidence in the central estimate. SARIMAX-X serves as the upside-risk scenario, target return 5.87%, for stress testing the portfolio under a higher-inflation regime.